# Noise simulation unit execution

Quick checks for the phenomenological NISQ-noise module.

In [2]:
import numpy as np
import os
import sys

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from noise.noise_simulation import (
    SimpleNISQNoiseParameters,
    amplitude_phase_damping_probabilities,
    amplitude_phase_damping_kraus,
    apply_amplitude_phase_damping,
    apply_default_noise_for_one_ancilla_step,
    computational_basis_probabilities,
    noisy_single_qubit_measurement,
)

In [3]:
# Example 1: T1/T2 -> channel parameters
gamma, lam = amplitude_phase_damping_probabilities(
    duration=200e-9, t1=80e-6, t2=60e-6
)
gamma, lam

(0.0024968776025399153, 0.004157998154890041)

In [9]:
# Example 2: apply amplitude+phase damping to |+><+|
plus = np.array([1.0, 1.0], dtype=complex) / np.sqrt(2)
rho_plus = np.outer(plus, plus.conj())
rho_noisy = apply_amplitude_phase_damping(
    rho_plus,
    target=0,
    num_qubits=1,
    duration=200e-9,
    t1=7e-6,
    t2=1e-6,
)
rho_noisy

array([[0.51408356+0.j, 0.33998239+0.j],
       [0.33998239+0.j, 0.48591644+0.j]])

In [7]:
# Example 3: default two-qubit layer for the one-ancilla workflow
psi = np.array([1, 0, 0, 1], dtype=complex) / np.sqrt(2)
rho_bell = np.outer(psi, psi.conj())
params = SimpleNISQNoiseParameters(
    t1_system=80e-6,
    t2_system=60e-6,
    t1_ancilla=70e-6,
    t2_ancilla=50e-6,
    two_qubit_gate_time=300e-9,
    p1q_depolarizing=5e-4,
    p2q_depolarizing=1e-2,
    readout_p0_to_1_system=0.02,
    readout_p1_to_0_system=0.03,
)
rho_bell_noisy = apply_default_noise_for_one_ancilla_step(rho_bell, params)
rho_bell_noisy

array([[0.49701289+0.j, 0.        +0.j, 0.        +0.j, 0.48520341+0.j],
       [0.        +0.j, 0.00510116+0.j, 0.        +0.j, 0.        +0.j],
       [0.        +0.j, 0.        +0.j, 0.0048374 +0.j, 0.        +0.j],
       [0.48520341+0.j, 0.        +0.j, 0.        +0.j, 0.49304855+0.j]])

In [10]:
# Example 4: classical readout error on a one-qubit Z-basis measurement
z_probs = np.array([1.0, 0.0])
meas = noisy_single_qubit_measurement(
    z_probs, p0_to_1=0.02, p1_to_0=0.04, shots=4000, rng=np.random.default_rng(7)
)
meas

NoisyMeasurementResult(ideal_probabilities=array([1., 0.]), observed_probabilities=array([0.98, 0.02]), assignment_matrix=array([[0.98, 0.04],
       [0.02, 0.96]]), counts={'0': 3913, '1': 87}, shots=4000)